# Load Data

In [ ]:
import pickle
import glob
import pandas as pd
import numpy as np
import urllib.request

In [ ]:
with open("Community_Detection_results.pkl", "rb") as f:
    Community_Detection_results = pickle.load(f)

In [ ]:
dfs = []

for path in glob.glob('*_AKE_results.pkl'):
    dataset_name = path.replace('_AKE_results.pkl', '')
    df = pd.read_pickle(path)
    df = df.drop(columns=['F', 'pF'])
    df = df.rename(columns={'Model': 'Method', 'hF': 'hF1'})
    df['Dataset'] = dataset_name
    dfs.append(df)

AKE_results = pd.concat(dfs, ignore_index=True)

In [ ]:
RESULTING_DATAFRAME = pd.merge(
    Community_Detection_results,
    AKE_results,
    on=['Dataset', 'Method'],
    how='left'
)

# Exhaustive Search

In [ ]:
RESULTING_DATAFRAME['C2'] = (1 + RESULTING_DATAFRAME['RI']) / 2
RESULTING_DATAFRAME['J'] = (
    RESULTING_DATAFRAME['hF1'] * RESULTING_DATAFRAME['C2'] * RESULTING_DATAFRAME['Modularity']
)

In [ ]:
RESULTING_DATAFRAME

,Dataset,Method,Measure,Algorithm,Modularity,Parameters,IP1,IP2,RI,hF1,C2,J
0,SemEval-2010,TF,CF,CNM,0.342335,[],0.0,0.0,0.0,0.134386,0.5,0.023003
1,SemEval-2010,TF,CF,Louvain,0.365538,[],0.0,0.0,0.0,0.134386,0.5,0.024562
2,SemEval-2010,TF,CF,Leiden,0.365538,[beta=0.2745941431317787],0.0,0.0,0.0,0.134386,0.5,0.024562
3,SemEval-2010,TF,CF,FLPA,0.000000,[],0.0,0.0,0.0,0.134386,0.5,0.000000
4,SemEval-2010,TF,Dice,CNM,0.333987,[],0.0,0.0,0.0,0.134386,0.5,0.022442
...,...,...,...,...,...,...,...,...,...,...,...,...
1083,500N-KP-Crowd,LMRANK,Jaccard,FLPA,0.344476,[],0.0,0.0,0.0,0.165640,0.5,0.028530
1084,500N-KP-Crowd,LMRANK,Cosine,CNM,0.400862,[],0.0,0.0,0.0,0.165640,0.5,0.033199
1085,500N-KP-Crowd,LMRANK,Cosine,Louvain,0.400862,[],0.0,0.0,0.0,0.165640,0.5,0.033199
1086,500N-KP-Crowd,LMRANK,Cosine,Leiden,0.401744,[beta=0.2745941431317787],0.0,0.0,0.0,0.165640,0.5,0.033272


In [ ]:
with open("RESULTING_DATAFRAME.pkl", "wb") as f:
    pickle.dump(RESULTING_DATAFRAME, f)

In [ ]:
target_dataset = 'WWW'
target_top_4 = RESULTING_DATAFRAME[RESULTING_DATAFRAME['Dataset'] == target_dataset].sort_values(by='J', ascending=False).head(4)
target_top_4 = target_top_4[['Dataset', 'Method', 'Measure', 'Algorithm', 'hF1', 'C2', 'Modularity', 'J']]

In [ ]:
target_top_4

,Dataset,Method,Measure,Algorithm,hF1,C2,Modularity,J
590,WWW,KPMiner,Cosine,Leiden,0.174705,0.681273,0.574928,0.068429
588,WWW,KPMiner,Cosine,CNM,0.174705,0.681273,0.574033,0.068322
589,WWW,KPMiner,Cosine,Louvain,0.174705,0.681273,0.570955,0.067956
586,WWW,KPMiner,Jaccard,Leiden,0.174705,0.690129,0.562312,0.067797


In [ ]:
top_4_across_datasets = RESULTING_DATAFRAME.sort_values(['Dataset', 'J'], ascending=[True, False]).groupby('Dataset').head(4)
top_4_across_datasets = top_4_across_datasets[['Dataset', 'Method', 'Measure', 'Algorithm', 'hF1', 'C2', 'Modularity', 'J']]

In [ ]:
top_4_across_datasets

,Dataset,Method,Measure,Algorithm,hF1,C2,Modularity,J
990,500N-KP-Crowd,TfIdf,Cosine,Leiden,0.336580,0.552853,0.570043,0.106073
989,500N-KP-Crowd,TfIdf,Cosine,Louvain,0.336580,0.552853,0.570043,0.106073
988,500N-KP-Crowd,TfIdf,Cosine,CNM,0.336580,0.552853,0.569823,0.106032
991,500N-KP-Crowd,TfIdf,Cosine,FLPA,0.336580,0.552853,0.533278,0.099232
956,DUC-2001,LMRANK,Cosine,CNM,0.240094,0.558488,0.623445,0.083597
957,DUC-2001,LMRANK,Cosine,Louvain,0.240094,0.558488,0.623445,0.083597
958,DUC-2001,LMRANK,Cosine,Leiden,0.240094,0.558488,0.623445,0.083597
954,DUC-2001,LMRANK,Jaccard,Leiden,0.240094,0.580848,0.579230,0.080778
378,Inspec,LMRANK,Jaccard,Leiden,0.363576,0.646053,0.535929,0.125884
376,Inspec,LMRANK,Jaccard,CNM,0.363576,0.646053,0.531011,0.124729


# Ablation Study

In [ ]:
datasets = RESULTING_DATAFRAME['Dataset'].unique()

for dataset in datasets:
    dataset_df = RESULTING_DATAFRAME[RESULTING_DATAFRAME['Dataset'] == dataset].copy()

    baseline_row = dataset_df[
        (dataset_df['Method'] == 'TF') &
        (dataset_df['Measure'] == 'CF') &
        (dataset_df['Algorithm'] == 'Louvain')
    ]

    base = baseline_row.iloc[0]
    best = dataset_df.sort_values(by='J', ascending=False).iloc[0]

    steps = [
        {'name': 'Baseline', 'm': base['Method'], 'e': base['Measure'], 'a': base['Algorithm']},
        {'name': 'Step 1', 'm': best['Method'], 'e': base['Measure'], 'a': base['Algorithm']},
        {'name': 'Step 2', 'm': best['Method'], 'e': best['Measure'], 'a': base['Algorithm']},
        {'name': 'Step 3', 'm': best['Method'], 'e': best['Measure'], 'a': best['Algorithm']}
    ]

    print(f"{'='*85}")
    print(f"Ablation Study ({dataset})")
    print(f"{'='*85}")
    print(f"{'Step':<10} | {'Policy':<40} | {'J (Improvement)':<25}")
    print(f"{'-'*85}")

    for step in steps:
        matched_row = dataset_df[
            (dataset_df['Method'] == step['m']) &
            (dataset_df['Measure'] == step['e']) &
            (dataset_df['Algorithm'] == step['a'])
        ]

        current = matched_row.iloc[0]
        difference_J = current['J'] - base['J']
        percentage_change_J = (difference_J / base['J'] * 100)

        policy = f"{current['Method']}-{current['Measure']}-{current['Algorithm']}"
        print(f"{step['name']:<10} | {policy:<40} | {current['J']:.3f} ({difference_J:+.3f}, {percentage_change_J:+.2f}%)")
        print(f"{' ':12} [hF1: {current['hF1']:.3f}, C2: {current['C2']:.3f}, Mod: {current['Modularity']:.3f}]")

Ablation Study (SemEval-2010)
Step       | Policy                                   | J (Improvement)          
-------------------------------------------------------------------------------------
Baseline   | TF-CF-Louvain                            | 0.025 (+0.000, +0.00%)
             [hF1: 0.134, C2: 0.500, Mod: 0.366]
Step 1     | KPMiner-CF-Louvain                       | 0.048 (+0.024, +96.70%)
             [hF1: 0.287, C2: 0.560, Mod: 0.300]
Step 2     | KPMiner-Cosine-Louvain                   | 0.072 (+0.047, +191.31%)
             [hF1: 0.287, C2: 0.530, Mod: 0.471]
Step 3     | KPMiner-Cosine-Leiden                    | 0.072 (+0.047, +192.87%)
             [hF1: 0.287, C2: 0.530, Mod: 0.473]
Ablation Study (NUS)
Step       | Policy                                   | J (Improvement)          
-------------------------------------------------------------------------------------
Baseline   | TF-CF-Louvain                            | 0.023 (+0.000, +0.00%)
             [hF1

# Extract Requirements

In [ ]:
!pip freeze > Optimization.txt